# Practice (pandas): ingesting messy orders, trap by trap

A batch pipeline over `data/input/customers.csv` and `data/input/orders.csv`, written as a walkthrough of the **traps** that ingestion has to handle.
The work is split into **stages**. Each stage has **steps**, and every step follows the same pattern:

> **Trap**: what goes wrong and how it shows in the data · **Handling**: the code that deals with it · **Test**: proof that the handling works, run right there

Each stage ends with a small function that wires its steps together. Stage E wires the stages into `run()`.
Types are explicit throughout:
- every column is read as text (A1)
- each output table has a declared schema (A3)
- every stage ends by enforcing its schema (B4, C9, D2)
- the outputs are read back with the same schema (E4)

Naming: there are two source tables, so frames are named after their table (`df_customers_raw`, `df_customers`, `df_orders_raw`, `df_orders_valid`, ...). A bare `df` appears only inside helpers that work on *any* table.

## Contents

- **[Stage A · Load the raw files and declare the types](#stage-a)**
  - [A1 · pandas guesses types and missing values](#a1)
  - [A2 · Rows lose track of where they came from](#a2)
  - [A3 · Types are guessed, lossy, or drift between runs](#a3)
  - [Stage A output: profile before you clean](#a-profile)
- **[Stage B · Clean customers](#stage-b)**
  - [B1 · Stray spaces, mixed case, blank vs missing](#b1)
  - [B2 · Dates in mixed formats](#b2)
  - [B3 · Several versions of one customer](#b3)
  - [B4 · Customer output types are not enforced](#b4)
- **[Stage C · Clean orders: valid vs quarantine](#stage-c)**
  - [C1 · One status spelt five ways](#c1)
  - [C2 · Exact duplicate rows](#c2)
  - [C3 · Missing vs unparseable values](#c3)
  - [C4 · Same `order_id`, different values](#c4)
  - [C5 · Values that break business rules](#c5)
  - [C6 · Orphan keys](#c6)
  - [C7 · Bad rows dropped silently](#c7)
  - [C8 · Join fan-out](#c8)
  - [C9 · Order output types are not enforced](#c9)
- **[Stage D · Aggregate daily revenue](#stage-d)**
  - [D1 · Cancelled orders counted as revenue](#d1)
  - [D2 · `groupby` drops NaN keys](#d2)
- **[Stage E · Publish](#stage-e)**
  - [E1 · Rows vanish between input and output](#e1)
  - [E2 · Nobody can see what was rejected](#e2)
  - [E3 · Reruns append or leave stale files](#e3)
  - [E4 · Types are lost when the output is written to CSV](#e4)
- **[All tests](#all-tests)**
- **[Notes (the kind of thing to write in the markdown cell they ask for)](#notes)**

Practise the **narration**: before each cell, say out loud what you expect; after it, say what surprised you.
Shortcuts: `Shift+Enter` run · `Esc` then `A`/`B` insert cell · `Esc` then `M`/`Y` markdown/code · `0 0` restart kernel · `Esc` then `X` cut cell.

In [1]:
import filecmp
import io
import json
import sys
import tempfile
import unittest
from pathlib import Path

import pandas as pd
from pandas.api import types as ptypes

print(sys.version.split()[0], pd.__version__)

# Repo root = the nearest folder with data/input, so this runs from the repo root or from marimo/jupyter/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "input").is_dir())
RAW = ROOT / "data" / "input"
OUT_DIR = ROOT / "data" / "output" / "practice_pandas"

VALID_STATUSES = {"shipped", "delivered", "refunded", "cancelled"}
CUSTOMER_COLUMNS = ["customer_id", "name", "email", "country", "signup_date", "updated_at"]
ORDER_COLUMNS = ["order_id", "customer_id", "order_date", "amount", "quantity", "status", "notes"]

ALL_TESTS: dict[str, type[unittest.TestCase]] = {}


def check(*cases: type[unittest.TestCase]) -> None:
    """Run these test classes now, and remember them for the full run at the end."""
    ALL_TESTS.update({c.__name__: c for c in cases})
    suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromTestCase(c) for c in cases)
    result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
    assert result.wasSuccessful(), "a test failed, see above"

3.12.11 3.1.1


---
## <a id="stage-a"></a>Stage A · Load the raw files and declare the types

Goal: get the files into memory **exactly as they arrived**, with every column as text. Then write down, as code, the types each output table must have, so later stages decide every type and every missing value on purpose.

### <a id="a1"></a>A1 · pandas guesses types and missing values

**Trap.** A plain `read_csv` infers types and treats some strings as missing. `customer_id` `00042` becomes the integer `42`, so it no longer matches `00042` elsewhere. Namibia's country code `"NA"` becomes NaN, along with `"N/A"` and `"null"`.

**Handling.** Read every column as text (`dtype=str`) and turn off the default missing-value list (`keep_default_na=False`). Blank cells come in as `""`, and the cleaning stages decide what is missing.

In [2]:
# See the trap: what a naive read does to the customers file
df_customers_naive = pd.read_csv(RAW / "customers.csv")
print(df_customers_naive.dtypes.astype(str).to_dict())
df_customers_naive[["customer_id", "country"]]

{'customer_id': 'int64', 'name': 'str', 'email': 'str', 'country': 'str', 'signup_date': 'str', 'updated_at': 'str'}


,customer_id,country
0,17,GB
1,42,gb
2,42,GB
3,73,ES
4,88,NaN
5,91,cn
6,91,cn
7,105,NaN


In [3]:
def read_text(path: Path) -> pd.DataFrame:
    """Read any CSV with every column as text and nothing turned into NaN."""
    return pd.read_csv(path, dtype=str, keep_default_na=False)

In [4]:
class TestA1NothingGuessed(unittest.TestCase):
    df_customers_text = read_text(RAW / "customers.csv")

    def test_every_column_is_text(self):
        for col in self.df_customers_text.columns:
            self.assertTrue(ptypes.is_string_dtype(self.df_customers_text[col]),
                            f"{col} is {self.df_customers_text[col].dtype}")

    def test_leading_zeros_preserved(self):
        self.assertIn("00042", set(self.df_customers_text["customer_id"]))

    def test_literal_NA_is_not_missing(self):
        self.assertIn("NA", set(self.df_customers_text["country"].str.strip()))
        self.assertFalse(self.df_customers_text.isna().any().any())


check(TestA1NothingGuessed)

test_every_column_is_text (__main__.TestA1NothingGuessed.test_every_column_is_text) ... 

ok


test_leading_zeros_preserved (__main__.TestA1NothingGuessed.test_leading_zeros_preserved) ... 

ok


test_literal_NA_is_not_missing (__main__.TestA1NothingGuessed.test_literal_NA_is_not_missing) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


### <a id="a2"></a>A2 · Rows lose track of where they came from

**Trap.** Once rows are filtered, sorted and deduplicated, nobody can point at the line in the file that caused a problem. Quarantine reports become useless.

**Handling.** Add `source_row`, the row's line number in the file. The header is line 1, so the first data row is line 2.

In [5]:
def add_source_row(df: pd.DataFrame) -> pd.DataFrame:
    """Works on any table: prepend the file line number of every row."""
    df = df.copy()
    df.insert(0, "source_row", range(2, len(df) + 2))
    return df


def read_raw(path: Path) -> pd.DataFrame:
    """Stage A: the file exactly as it arrived (all text) plus its line numbers."""
    return add_source_row(read_text(path))


df_customers_raw = read_raw(RAW / "customers.csv")
df_orders_raw = read_raw(RAW / "orders.csv")
print("customers", df_customers_raw.shape, "| orders", df_orders_raw.shape)
df_orders_raw.head()

customers (8, 7) | orders (20, 8)


,source_row,order_id,customer_id,order_date,amount,quantity,status,notes
0,2,1001,00017,2024-03-01,12.50,1,Shipped,
1,3,1002,00042,2024-03-01,"£1,200.00",2,shipped,"gift wrap, urgent"
2,4,1003,00073,2024-03-02,30,3,DELIVERED,
3,5,1004,00088,2024-03-02,45.00,1,delivered,
4,6,1005,00091,2024-02-30,20.00,1,shipped,


In [6]:
class TestA2Lineage(unittest.TestCase):
    path = RAW / "orders.csv"
    df_orders_raw = read_raw(path)
    lines = path.read_text(encoding="utf-8").splitlines()

    def test_row_count_matches_file(self):
        self.assertEqual(len(self.df_orders_raw), len(self.lines) - 1)

    def test_source_row_points_at_the_line(self):
        src = self.df_orders_raw.set_index("order_id").loc["1007", "source_row"]
        self.assertTrue(self.lines[src - 1].startswith("1007,"))

    def test_source_row_unique(self):
        self.assertTrue(self.df_orders_raw["source_row"].is_unique)


check(TestA2Lineage)

test_row_count_matches_file (__main__.TestA2Lineage.test_row_count_matches_file) ... 

ok


test_source_row_points_at_the_line (__main__.TestA2Lineage.test_source_row_points_at_the_line) ... 

ok


test_source_row_unique (__main__.TestA2Lineage.test_source_row_unique) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


### <a id="a3"></a>A3 · Types are guessed, lossy, or drift between runs

**Trap.** Even after a text-only load, types can go wrong *later*, and always silently:
1. a single missing value turns an integer column into `float` (`1` becomes `1.0`)
2. `int64` cannot hold a missing value at all, so the cast crashes
3. `float` money does not add up exactly (`0.1 + 0.2`)
4. a plain text column accepts any status, `lost_in_space` included
5. pandas 3 parses dates as `datetime64[us]`, pandas 2 as `datetime64[ns]`: the same code gives a different schema on another version
6. a column that happens to be all blank today is read back as `float64`

**Handling.** Declare the schema of every **output** table as a dict of column → dtype, and enforce it with one function:
- select and order exactly the declared columns
- refuse values outside a categorical's allowed set
- cast to the declared dtype, and fail loudly on unsafe casts

`schema_diff` is the matching test helper. Choices made here:

| kind | dtype | why |
|---|---|---|
| ids (`customer_id`, `order_id`) | `str` | keep leading zeros; ids are labels, not numbers |
| free text, country | `str` | pandas 3 default string type; missing = NaN |
| dates / timestamps | `datetime64[ns]` | one resolution, pinned explicitly |
| `amount`, `revenue` | `float64`, rounded to 2 dp | simple. Integer pence or `Decimal` would be exact, which is worth naming as the next step |
| `quantity` | `Int64` (capital I) | nullable integer: whole numbers, and a missing value stays `<NA>` instead of becoming float |
| `status` | `category` of the allowed set | the domain is part of the type |
| `source_row`, `orders` | `int64` | never missing |

In [7]:
# See the trap: five ways types go wrong without an error
qty = pd.to_numeric(pd.Series(["1", "", "3"]), errors="coerce")
print("1. one blank turns ints into floats:", qty.tolist(), qty.dtype)
try:
    qty.astype("int64")
except ValueError as e:
    print("2. int64 cannot hold a missing value:", type(e).__name__)
print("   nullable Int64 can:", qty.astype("Int64").tolist())
print("3. float money:", 0.1 + 0.2, "| rounded to pennies:", round(0.1 + 0.2, 2))
print("4. text accepts any status:", pd.Series(["shipped", "lost_in_space"]).dtype)
print("5. default date resolution in this pandas:", pd.to_datetime(pd.Series(["2024-03-01"])).dtype)
print("6. an all-blank column read back:", pd.read_csv(io.StringIO("order_id,notes\n1,\n2,\n")).dtypes.to_dict())

1. one blank turns ints into floats: [1.0, nan, 3.0] float64
2. int64 cannot hold a missing value: IntCastingNaNError
   nullable Int64 can: [1, <NA>, 3]
3. float money: 0.30000000000000004 | rounded to pennies: 0.3
4. text accepts any status: str
5. default date resolution in this pandas: datetime64[us]
6. an all-blank column read back: {'order_id': dtype('int64'), 'notes': dtype('float64')}


In [8]:
STATUS_DTYPE = pd.CategoricalDtype(sorted(VALID_STATUSES))

CUSTOMER_SCHEMA = {
    "customer_id": "str",
    "name": "str",
    "email": "str",
    "country": "str",
    "signup_date": "datetime64[ns]",
    "updated_at": "datetime64[ns]",
}
ORDER_SCHEMA = {
    "order_id": "str",
    "customer_id": "str",
    "country": "str",
    "order_date": "datetime64[ns]",
    "amount": "float64",
    "quantity": "Int64",
    "status": STATUS_DTYPE,
    "notes": "str",
    "source_row": "int64",
}
DAILY_REVENUE_SCHEMA = {
    "order_date": "datetime64[ns]",
    "country": "str",
    "revenue": "float64",
    "orders": "int64",
}


def enforce_schema(df: pd.DataFrame, schema: dict) -> pd.DataFrame:
    """Works on any table: exactly the declared columns, in order, cast to the declared dtypes.

    Raises instead of guessing: a missing column, a value outside a categorical's
    allowed set, or an unsafe cast (1.5 -> Int64, NaN -> int64) all fail.
    """
    missing = [col for col in schema if col not in df.columns]
    if missing:
        raise KeyError(f"missing columns: {missing}")
    df = df[list(schema)].copy()                       # drops undeclared columns, fixes the order
    for col, dtype in schema.items():
        if isinstance(dtype, pd.CategoricalDtype):     # astype would silently turn unknowns into NaN
            unknown = set(df[col].dropna()) - set(dtype.categories)
            if unknown:
                raise TypeError(f"{col}: values outside {list(dtype.categories)}: {sorted(unknown)}")
        df[col] = df[col].astype(dtype)
    return df


def schema_diff(df: pd.DataFrame, schema: dict) -> dict:
    """Test helper: {column: (expected, actual)} for every mismatch; {} means the frame matches."""
    diff = {col: (str(dtype), str(df[col].dtype)) for col, dtype in schema.items()
            if col in df.columns and df[col].dtype != dtype}
    diff |= {col: (str(dtype), "missing") for col, dtype in schema.items() if col not in df.columns}
    diff |= {col: ("not in schema", str(df[col].dtype)) for col in df.columns if col not in schema}
    if not diff and list(df.columns) != list(schema):
        diff["<column order>"] = (list(schema), list(df.columns))
    return diff

In [9]:
class TestA3Schema(unittest.TestCase):
    schema = {"order_id": "str", "quantity": "Int64", "status": STATUS_DTYPE}

    def fixture(self, **overrides):
        df_orders = pd.DataFrame({"status": ["shipped", "refunded"], "order_id": ["00042", "7"],
                                  "quantity": [1.0, None], "extra": ["x", "y"]})
        return df_orders.assign(**overrides)

    def test_columns_selected_ordered_and_cast(self):
        df_orders = enforce_schema(self.fixture(), self.schema)
        self.assertEqual(list(df_orders.columns), ["order_id", "quantity", "status"])
        self.assertEqual(schema_diff(df_orders, self.schema), {})

    def test_ids_stay_text(self):
        self.assertEqual(enforce_schema(self.fixture(), self.schema)["order_id"].iloc[0], "00042")

    def test_missing_quantity_stays_integer(self):
        quantity = enforce_schema(self.fixture(), self.schema)["quantity"]
        self.assertEqual(str(quantity.dtype), "Int64")
        self.assertTrue(pd.isna(quantity.iloc[1]))

    def test_missing_column_raises(self):
        with self.assertRaises(KeyError):
            enforce_schema(self.fixture().drop(columns="quantity"), self.schema)

    def test_unknown_category_raises(self):
        with self.assertRaises(TypeError):
            enforce_schema(self.fixture(status=["shipped", "lost_in_space"]), self.schema)

    def test_fractional_quantity_raises(self):
        with self.assertRaises(TypeError):
            enforce_schema(self.fixture(quantity=[1.5, 2.0]), self.schema)

    def test_schema_diff_reports_mismatches(self):
        diff = schema_diff(self.fixture(), self.schema)
        self.assertEqual(diff["quantity"], ("Int64", "float64"))
        self.assertEqual(diff["extra"][0], "not in schema")


check(TestA3Schema)

test_columns_selected_ordered_and_cast (__main__.TestA3Schema.test_columns_selected_ordered_and_cast) ... 

ok


test_fractional_quantity_raises (__main__.TestA3Schema.test_fractional_quantity_raises) ... 

ok


test_ids_stay_text (__main__.TestA3Schema.test_ids_stay_text) ... 

ok


... (5 outputs left out)

ok


test_unknown_category_raises (__main__.TestA3Schema.test_unknown_category_raises) ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.006s

OK


### <a id="a-profile"></a>Stage A output: profile before you clean

Not a trap, but the scouting step: it shows which traps are waiting in stages B and C.

In [10]:
print("blank cells:\n", df_orders_raw.eq("").sum()[lambda s: s > 0], sep="")
print("\nstatus values:", df_orders_raw["status"].value_counts().to_dict())
for col, conv in {"amount": lambda s: pd.to_numeric(s, errors="coerce"),
                  "quantity": lambda s: pd.to_numeric(s, errors="coerce"),
                  "order_date": lambda s: pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")}.items():
    print(f"{col:11s} won't convert: {df_orders_raw.loc[conv(df_orders_raw[col]).isna(), col].unique().tolist()}")
print("\nrows sharing an order_id:")
df_orders_raw[df_orders_raw["order_id"].duplicated(keep=False)]

blank cells:
order_date     1
quantity       1
notes         17
dtype: int64

status values: {'shipped': 10, 'Shipped': 3, 'shipped ': 1, 'DELIVERED': 1, 'delivered': 1, 'refunded': 1, 'cancelled': 1, 'lost_in_space': 1, 'Cancelled': 1}
amount      won't convert: ['£1,200.00', 'N/A']
quantity    won't convert: ['', 'two']
order_date  won't convert: ['2024-02-30', '', '06/03/2024']

rows sharing an order_id:


,source_row,order_id,customer_id,order_date,amount,quantity,status,notes
11,13,1012,00042,2024-03-05,7.25,1,Shipped,
12,14,1012,00042,2024-03-05,7.25,1,Shipped,
13,15,1013,00073,2024-03-05,100.00,1,shipped,
14,16,1013,00073,2024-03-05,110.00,1,shipped,amended?


---
## <a id="stage-b"></a>Stage B · Clean customers

Goal: one tidy row per `customer_id` that matches `CUSTOMER_SCHEMA`, so stage C can check orders against it and join to it.

### <a id="b1"></a>B1 · Stray spaces, mixed case, blank vs missing

**Trap.** `"  Alice Smith "`, `"ALICE@example.com"`, `"gb "` and `" cn"` look different to a computer but mean the same thing. Blank cells are `""` (from A1), which is not the same as missing.

**Handling.** Strip every column. Lower-case emails and upper-case country codes. Then turn blanks into NA. This runs *after* A1, so `"NA"` is still the text Namibia and is not affected.

In [11]:
def _blank_to_na(s: pd.Series) -> pd.Series:
    return s.mask(s.eq(""))


def tidy_customer_text(df_customers: pd.DataFrame) -> pd.DataFrame:
    df_customers = df_customers.copy()
    for col in CUSTOMER_COLUMNS:
        df_customers[col] = df_customers[col].str.strip()
    df_customers["email"] = df_customers["email"].str.lower()
    df_customers["country"] = df_customers["country"].str.upper()
    for col in ["name", "email", "country"]:
        df_customers[col] = _blank_to_na(df_customers[col])
    return df_customers


df_customers_tidy = tidy_customer_text(df_customers_raw)
df_customers_tidy

,source_row,customer_id,name,email,country,signup_date,updated_at
0,2,00017,Alice Smith,alice@example.com,GB,2023-01-15,2024-01-01T09:00:00
1,3,00042,Bob Jones,bob@example.com,GB,2023-02-20,2024-01-01T09:00:00
2,4,00042,Bob Jones,robert.jones@example.com,GB,2023-02-20,2024-03-10T12:30:00
3,5,00073,José Núñez,jose@example.es,ES,2023-03-05,2024-01-01T09:00:00
4,6,00088,Nangula Shikongo,nangula@example.na,NA,2023-04-11,2024-01-01T09:00:00
5,7,00091,Chen Wei,chen@example.cn,CN,05/06/2023,2024-01-01T09:00:00
6,8,00091,Chen Wei,chen@example.cn,CN,05/06/2023,2024-01-01T09:00:00
7,9,00105,NaN,anon@example.com,NaN,2023-07-01,2024-01-01T09:00:00


In [12]:
class TestB1TidyText(unittest.TestCase):
    df_customers = tidy_customer_text(read_raw(RAW / "customers.csv"))

    def test_no_surrounding_whitespace(self):
        for col in ["name", "email", "country"]:
            self.assertFalse(self.df_customers[col].dropna().str.match(r"^\s|.*\s$").any(), col)

    def test_case_normalised(self):
        emails = self.df_customers["email"].dropna()
        self.assertTrue((emails == emails.str.lower()).all())
        self.assertEqual(set(self.df_customers["country"].dropna()), {"GB", "ES", "NA", "CN"})

    def test_blank_becomes_missing(self):
        anon = self.df_customers.set_index("customer_id").loc["00105"]
        self.assertTrue(pd.isna(anon["name"]) and pd.isna(anon["country"]))


check(TestB1TidyText)

test_blank_becomes_missing (__main__.TestB1TidyText.test_blank_becomes_missing) ... 

ok


test_case_normalised (__main__.TestB1TidyText.test_case_normalised) ... 

ok


test_no_surrounding_whitespace (__main__.TestB1TidyText.test_no_surrounding_whitespace) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


### <a id="b2"></a>B2 · Dates in mixed formats

**Trap.** `05/06/2023` is 5 June in the UK and 6 May in the US. If pandas is allowed to guess, it picks one without telling you, and impossible dates may be rolled over or mixed in.

**Handling.** Parse with an explicit `format=` and `errors="coerce"`. Anything that does not match becomes `NaT`, so it is visible rather than guessed.

In [13]:
# See the trap: the same text gives two different dates depending on a flag
s = pd.Series(["05/06/2023"])
print(pd.to_datetime(s, format="mixed").iloc[0].date(), "vs",
      pd.to_datetime(s, format="mixed", dayfirst=True).iloc[0].date())

2023-05-06 vs 2023-06-05


In [14]:
def parse_customer_dates(df_customers: pd.DataFrame) -> pd.DataFrame:
    df_customers = df_customers.copy()
    df_customers["signup_date"] = pd.to_datetime(df_customers["signup_date"], format="%Y-%m-%d", errors="coerce")
    df_customers["updated_at"] = pd.to_datetime(df_customers["updated_at"], format="%Y-%m-%dT%H:%M:%S", errors="coerce")
    return df_customers

In [15]:
class TestB2StrictDates(unittest.TestCase):
    df_customers = parse_customer_dates(tidy_customer_text(read_raw(RAW / "customers.csv")))

    def test_dates_are_datetimes(self):
        self.assertTrue(ptypes.is_datetime64_any_dtype(self.df_customers["signup_date"]))
        self.assertTrue(ptypes.is_datetime64_any_dtype(self.df_customers["updated_at"]))

    def test_iso_date_parsed(self):
        alice = self.df_customers.set_index("customer_id").loc["00017"]
        self.assertEqual(alice["signup_date"], pd.Timestamp("2023-01-15"))

    def test_ambiguous_date_not_guessed(self):
        chen = self.df_customers.loc[self.df_customers["customer_id"] == "00091", "signup_date"]
        self.assertTrue(chen.isna().all())


check(TestB2StrictDates)

test_ambiguous_date_not_guessed (__main__.TestB2StrictDates.test_ambiguous_date_not_guessed) ... 

ok


test_dates_are_datetimes (__main__.TestB2StrictDates.test_dates_are_datetimes) ... 

ok


test_iso_date_parsed (__main__.TestB2StrictDates.test_iso_date_parsed) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


### <a id="b3"></a>B3 · Several versions of one customer

**Trap.** `00042` appears twice with different emails, and `00091` appears twice as an exact copy. Left in, every join on `customer_id` would duplicate orders (see C8).

**Handling.** Sort by `updated_at` (with `source_row` as a tie-break) and keep the last row per id, so the latest version wins whatever the file order. Exact copies collapse the same way.

In [16]:
def latest_customer_version(df_customers: pd.DataFrame) -> pd.DataFrame:
    return (
        df_customers.sort_values(["customer_id", "updated_at", "source_row"])
        .drop_duplicates(subset="customer_id", keep="last")
        .reset_index(drop=True)
    )


latest_customer_version(parse_customer_dates(df_customers_tidy))

,source_row,customer_id,name,email,country,signup_date,updated_at
0,2,00017,Alice Smith,alice@example.com,GB,2023-01-15,2024-01-01 09:00:00
1,4,00042,Bob Jones,robert.jones@example.com,GB,2023-02-20,2024-03-10 12:30:00
2,5,00073,José Núñez,jose@example.es,ES,2023-03-05,2024-01-01 09:00:00
3,6,00088,Nangula Shikongo,nangula@example.na,NA,2023-04-11,2024-01-01 09:00:00
4,8,00091,Chen Wei,chen@example.cn,CN,NaT,2024-01-01 09:00:00
5,9,00105,NaN,anon@example.com,NaN,2023-07-01,2024-01-01 09:00:00


In [17]:
class TestB3LatestVersion(unittest.TestCase):
    df_customers = latest_customer_version(parse_customer_dates(tidy_customer_text(read_raw(RAW / "customers.csv"))))

    def test_customer_id_unique(self):
        self.assertTrue(self.df_customers["customer_id"].is_unique)
        self.assertEqual(len(self.df_customers), 6)

    def test_latest_version_wins(self):
        email = self.df_customers.set_index("customer_id").loc["00042", "email"]
        self.assertEqual(email, "robert.jones@example.com")

    def test_file_order_does_not_matter(self):
        # the newer version comes *first* in the file
        df_customers = pd.DataFrame({
            "customer_id": ["1", "1"],
            "updated_at": pd.to_datetime(["2024-05-01", "2024-01-01"]),
            "source_row": [2, 3],
            "email": ["new", "old"],
        })
        self.assertEqual(latest_customer_version(df_customers)["email"].tolist(), ["new"])


check(TestB3LatestVersion)

test_customer_id_unique (__main__.TestB3LatestVersion.test_customer_id_unique) ... 

ok


test_file_order_does_not_matter (__main__.TestB3LatestVersion.test_file_order_does_not_matter) ... 

ok


test_latest_version_wins (__main__.TestB3LatestVersion.test_latest_version_wins) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


### <a id="b4"></a>B4 · Customer output types are not enforced

**Trap.** B1–B3 *happen* to produce sensible types, but nothing guarantees them. A refactor, a pandas upgrade (`[us]` vs `[ns]`) or a column that is all blank today changes the output schema, and the join in C8 or the database load breaks later, far from the cause.

**Handling.** `clean_customers` ends with `enforce_schema(..., CUSTOMER_SCHEMA)`. The stage's output type is a contract.

In [18]:
def clean_customers(df_customers_raw: pd.DataFrame) -> pd.DataFrame:
    """Stage B: tidy text, strict dates, one row per customer, declared types."""
    df_customers = tidy_customer_text(df_customers_raw)        # B1
    df_customers = parse_customer_dates(df_customers)          # B2
    df_customers = latest_customer_version(df_customers)       # B3
    return enforce_schema(df_customers, CUSTOMER_SCHEMA)       # B4


print("before B4:", schema_diff(latest_customer_version(parse_customer_dates(df_customers_tidy)), CUSTOMER_SCHEMA))
df_customers = clean_customers(df_customers_raw)
print("after B4: ", schema_diff(df_customers, CUSTOMER_SCHEMA))
df_customers.dtypes

before B4: {'signup_date': ('datetime64[ns]', 'datetime64[us]'), 'updated_at': ('datetime64[ns]', 'datetime64[us]'), 'source_row': ('not in schema', 'int64')}
after B4:  {}


customer_id               str
name                      str
email                     str
country                   str
signup_date    datetime64[ns]
updated_at     datetime64[ns]
dtype: object

In [19]:
class TestB4CustomerSchema(unittest.TestCase):
    df_customers = clean_customers(read_raw(RAW / "customers.csv"))

    def test_matches_schema_exactly(self):
        self.assertEqual(schema_diff(self.df_customers, CUSTOMER_SCHEMA), {})

    def test_values_survive_the_cast(self):
        by_id = self.df_customers.set_index("customer_id")
        self.assertEqual(by_id.loc["00088", "country"], "NA")                         # Namibia, not NaN
        self.assertEqual(by_id.loc["00042", "updated_at"], pd.Timestamp("2024-03-10 12:30:00"))
        self.assertTrue(pd.isna(by_id.loc["00091", "signup_date"]))                    # NaT stays NaT


check(TestB4CustomerSchema)

test_matches_schema_exactly (__main__.TestB4CustomerSchema.test_matches_schema_exactly) ... 

ok


test_values_survive_the_cast (__main__.TestB4CustomerSchema.test_values_survive_the_cast) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


---
## <a id="stage-c"></a>Stage C · Clean orders: valid vs quarantine

Goal: split every order into **valid** (typed per `ORDER_SCHEMA` and joined to its customer) or **quarantine** (raw text plus a reason). Nothing is silently dropped.

Each check is a *rule*: a function `(df_orders, df_orders_typed, df_customers) -> {reason: boolean mask}`. Every rule has the same signature, so `clean_orders` can loop over them, and each one can be tested on its own.

In [20]:
def flagged(df_orders: pd.DataFrame, checks: dict[str, pd.Series]) -> dict[str, set[str]]:
    """Test helper: which order_ids each rule flags."""
    return {name: set(df_orders.loc[mask, "order_id"]) for name, mask in checks.items()}

### <a id="c1"></a>C1 · One status spelt five ways

**Trap.** `Shipped`, `shipped `, `SHIPPED` are one status. Left as they are, they fail the allowed-status check, and they also hide real duplicates (C2).

**Handling.** Strip every column and lower-case `status`. This has to run **before** deduplication.

In [21]:
def normalise_orders(df_orders: pd.DataFrame) -> pd.DataFrame:
    df_orders = df_orders.copy()
    for col in ORDER_COLUMNS:
        df_orders[col] = df_orders[col].str.strip()
    df_orders["status"] = df_orders["status"].str.lower()
    return df_orders


df_orders_norm = normalise_orders(df_orders_raw)
df_orders_norm["status"].value_counts().to_dict()

{'shipped': 14,
 'delivered': 2,
 'cancelled': 2,
 'refunded': 1,
 'lost_in_space': 1}

In [22]:
class TestC1Normalise(unittest.TestCase):
    df_orders = normalise_orders(read_raw(RAW / "orders.csv"))

    def test_status_spellings_collapse(self):
        self.assertEqual(set(self.df_orders["status"]), VALID_STATUSES | {"lost_in_space"})

    def test_no_surrounding_whitespace(self):
        for col in ORDER_COLUMNS:
            self.assertFalse(self.df_orders[col].str.match(r"^\s|.*\s$").any(), col)


check(TestC1Normalise)

test_no_surrounding_whitespace (__main__.TestC1Normalise.test_no_surrounding_whitespace) ... 

ok


test_status_spellings_collapse (__main__.TestC1Normalise.test_status_spellings_collapse) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


### <a id="c2"></a>C2 · Exact duplicate rows

**Trap.** Order `1012` arrives twice with identical values (a resend or double export). Counting it twice doubles its revenue.

**Handling.** Drop rows that match on every business column, keep the first, and **count** how many were dropped, for the reconciliation in E1. `source_row` is left out of the comparison because it always differs.

In [23]:
def drop_exact_duplicates(df_orders: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    dup = df_orders.duplicated(subset=ORDER_COLUMNS, keep="first")
    return df_orders.loc[~dup], int(dup.sum())


df_orders_dedup, n_exact = drop_exact_duplicates(df_orders_norm)
print("exact duplicates dropped:", n_exact)

exact duplicates dropped: 1


In [24]:
class TestC2ExactDuplicates(unittest.TestCase):
    df_orders, n = drop_exact_duplicates(normalise_orders(read_raw(RAW / "orders.csv")))

    def test_one_dropped_and_counted(self):
        self.assertEqual(self.n, 1)
        self.assertEqual((self.df_orders["order_id"] == "1012").sum(), 1)

    def test_first_copy_kept(self):
        self.assertEqual(self.df_orders.set_index("order_id").loc["1012", "source_row"], 13)

    def test_conflicting_rows_not_treated_as_exact(self):
        self.assertEqual((self.df_orders["order_id"] == "1013").sum(), 2)  # left for C4


check(TestC2ExactDuplicates)

test_conflicting_rows_not_treated_as_exact (__main__.TestC2ExactDuplicates.test_conflicting_rows_not_treated_as_exact) ... 

ok


test_first_copy_kept (__main__.TestC2ExactDuplicates.test_first_copy_kept) ... 

ok


test_one_dropped_and_counted (__main__.TestC2ExactDuplicates.test_one_dropped_and_counted) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


### <a id="c3"></a>C3 · Missing vs unparseable values

**Trap.** `N/A` in `amount`, `two` in `quantity`, `2024-02-30` and `06/03/2024` in `order_date`, and `£1,200.00` as an amount. `errors="coerce"` turns all of them into NaN, so a blank cell and a broken one look the same. Without care, `£1,200.00` is lost even though it is perfectly readable.

**Handling.** Strip currency symbols and thousands separators, then parse strictly. Compare the result with the raw text: blank text means *missing*, non-blank text that failed means *unparseable*. Two different reasons, two different fixes upstream.

In [25]:
def parse_orders(df_orders: pd.DataFrame) -> pd.DataFrame:
    """Typed versions of the text columns. Failures become NaN/NaT, never a guess."""
    return pd.DataFrame({
        "order_date": pd.to_datetime(df_orders["order_date"], format="%Y-%m-%d", errors="coerce"),
        "amount": pd.to_numeric(df_orders["amount"].str.replace(r"[£$€,]", "", regex=True), errors="coerce"),
        "quantity": pd.to_numeric(df_orders["quantity"], errors="coerce"),
    }, index=df_orders.index)


def rules_parsing(df_orders, df_orders_typed, df_customers) -> dict[str, pd.Series]:
    checks = {}
    for col in ["order_date", "amount", "quantity"]:
        checks[f"missing_{col}"] = df_orders[col].eq("")
        checks[f"unparseable_{col}"] = df_orders[col].ne("") & df_orders_typed[col].isna()
    return checks


df_orders_typed = parse_orders(df_orders_dedup)
print(df_orders_typed.dtypes.astype(str).to_dict())
flagged(df_orders_dedup, rules_parsing(df_orders_dedup, df_orders_typed, df_customers))

{'order_date': 'datetime64[us]', 'amount': 'float64', 'quantity': 'float64'}


{'missing_order_date': {'1009'},
 'unparseable_order_date': {'1005', '1016'},
 'missing_amount': set(),
 'unparseable_amount': {'1006'},
 'missing_quantity': {'1010'},
 'unparseable_quantity': {'1011'}}

In [26]:
class TestC3Parsing(unittest.TestCase):
    df_orders, _ = drop_exact_duplicates(normalise_orders(read_raw(RAW / "orders.csv")))
    df_orders_typed = parse_orders(df_orders)

    def test_each_order_gets_the_right_reason(self):
        self.assertEqual(flagged(self.df_orders, rules_parsing(self.df_orders, self.df_orders_typed, None)), {
            "missing_order_date": {"1009"},
            "unparseable_order_date": {"1005", "1016"},  # 30 February; ambiguous dd/mm
            "missing_amount": set(),
            "unparseable_amount": {"1006"},              # "N/A"
            "missing_quantity": {"1010"},
            "unparseable_quantity": {"1011"},            # "two"
        })

    def test_currency_text_parsed(self):
        amount = self.df_orders_typed.loc[self.df_orders["order_id"] == "1002", "amount"].item()
        self.assertEqual(amount, 1200.00)

    def test_typed_columns(self):
        self.assertTrue(ptypes.is_datetime64_any_dtype(self.df_orders_typed["order_date"]))
        self.assertTrue(ptypes.is_float_dtype(self.df_orders_typed["amount"]))


check(TestC3Parsing)

test_currency_text_parsed (__main__.TestC3Parsing.test_currency_text_parsed) ... 

ok


test_each_order_gets_the_right_reason (__main__.TestC3Parsing.test_each_order_gets_the_right_reason) ... 

ok


test_typed_columns (__main__.TestC3Parsing.test_typed_columns) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


### <a id="c4"></a>C4 · Same `order_id`, different values

**Trap.** Order `1013` arrives twice, once at £100 and once at £110 (`"amended?"`). Keeping the first or last copy is a guess, and a wrong guess changes revenue.

**Handling.** Flag **both** rows so a person decides. It has to run after C2, or exact copies like `1012` would be flagged as conflicts too.

In [27]:
def rules_duplicates(df_orders, df_orders_typed, df_customers) -> dict[str, pd.Series]:
    return {"conflicting_duplicate_order_id": df_orders["order_id"].duplicated(keep=False)}


flagged(df_orders_dedup, rules_duplicates(df_orders_dedup, df_orders_typed, df_customers))

{'conflicting_duplicate_order_id': {'1013'}}

In [28]:
class TestC4ConflictingDuplicates(unittest.TestCase):
    df_orders_norm = normalise_orders(read_raw(RAW / "orders.csv"))
    df_orders, _ = drop_exact_duplicates(df_orders_norm)

    def test_both_copies_flagged(self):
        mask = rules_duplicates(self.df_orders, None, None)["conflicting_duplicate_order_id"]
        self.assertEqual(set(self.df_orders.loc[mask, "order_id"]), {"1013"})
        self.assertEqual(mask.sum(), 2)

    def test_order_matters_dedup_first(self):
        # run on the un-deduplicated frame, 1012 would be wrongly called a conflict
        mask = rules_duplicates(self.df_orders_norm, None, None)["conflicting_duplicate_order_id"]
        self.assertIn("1012", set(self.df_orders_norm.loc[mask, "order_id"]))


check(TestC4ConflictingDuplicates)

test_both_copies_flagged (__main__.TestC4ConflictingDuplicates.test_both_copies_flagged) ... 

ok


test_order_matters_dedup_first (__main__.TestC4ConflictingDuplicates.test_order_matters_dedup_first) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


### <a id="c5"></a>C5 · Values that break business rules

**Trap.** A value can parse cleanly and still be wrong. `lost_in_space` is not a status. A negative amount only makes sense for a refund. A quantity of `0` or `1.5` parses as a number but is not a real order, and `1.5` would crash the cast to `Int64` in C9.

**Handling.** Check statuses against an allowed set, accept negative amounts only when `status == "refunded"`, and require whole, positive quantities.

In [29]:
def rules_business(df_orders, df_orders_typed, df_customers) -> dict[str, pd.Series]:
    qty = df_orders_typed["quantity"]
    return {
        "invalid_status": ~df_orders["status"].isin(VALID_STATUSES),
        "negative_amount": df_orders_typed["amount"].lt(0) & df_orders["status"].ne("refunded"),
        "invalid_quantity": qty.notna() & (qty.le(0) | qty.mod(1).ne(0)),
    }


flagged(df_orders_dedup, rules_business(df_orders_dedup, df_orders_typed, df_customers))

{'invalid_status': {'1015'},
 'negative_amount': set(),
 'invalid_quantity': set()}

In [30]:
class TestC5BusinessRules(unittest.TestCase):
    df_orders, _ = drop_exact_duplicates(normalise_orders(read_raw(RAW / "orders.csv")))

    def test_sample_file(self):
        checks = rules_business(self.df_orders, parse_orders(self.df_orders), None)
        self.assertEqual(flagged(self.df_orders, checks), {
            "invalid_status": {"1015"},
            "negative_amount": set(),  # 1008 is -5.00 but refunded
            "invalid_quantity": set(),
        })

    def test_cases_not_in_the_sample(self):
        df_orders_fx = pd.DataFrame({
            "order_id": ["neg_ship", "neg_refund", "zero_qty", "half_qty"],
            "order_date": ["2024-03-01"] * 4,
            "amount": ["-5", "-5", "5", "5"],
            "quantity": ["1", "1", "0", "1.5"],
            "status": ["shipped", "refunded", "shipped", "shipped"],
        })
        f = flagged(df_orders_fx, rules_business(df_orders_fx, parse_orders(df_orders_fx), None))
        self.assertEqual(f["negative_amount"], {"neg_ship"})
        self.assertEqual(f["invalid_quantity"], {"zero_qty", "half_qty"})


check(TestC5BusinessRules)

test_cases_not_in_the_sample (__main__.TestC5BusinessRules.test_cases_not_in_the_sample) ... 

ok


test_sample_file (__main__.TestC5BusinessRules.test_sample_file) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.003s

OK


### <a id="c6"></a>C6 · Orphan keys

**Trap.** Order `1007` belongs to customer `00999`, who does not exist. Order `1014` has customer `17`: probably `00017` with its zeros lost in Excel, but padding it is a guess that could put revenue on the wrong customer.

**Handling.** Reject any `customer_id` that is not in the cleaned customer master. No padding, no fuzzy matching. Both sides are `str` (A1, B4), so `"00017"` is never compared with the number `17`.

In [31]:
def rules_reference(df_orders, df_orders_typed, df_customers) -> dict[str, pd.Series]:
    return {"unknown_customer": ~df_orders["customer_id"].isin(df_customers["customer_id"])}


flagged(df_orders_dedup, rules_reference(df_orders_dedup, df_orders_typed, df_customers))

{'unknown_customer': {'1007', '1014'}}

In [32]:
class TestC6OrphanKeys(unittest.TestCase):
    df_orders, _ = drop_exact_duplicates(normalise_orders(read_raw(RAW / "orders.csv")))
    df_customers = clean_customers(read_raw(RAW / "customers.csv"))

    def test_unknown_customers_flagged(self):
        f = flagged(self.df_orders, rules_reference(self.df_orders, None, self.df_customers))
        self.assertEqual(f["unknown_customer"], {"1007", "1014"})

    def test_unpadded_id_not_guessed(self):
        self.assertEqual(self.df_orders.set_index("order_id").loc["1014", "customer_id"], "17")


check(TestC6OrphanKeys)

test_unknown_customers_flagged (__main__.TestC6OrphanKeys.test_unknown_customers_flagged) ... 

ok


test_unpadded_id_not_guessed (__main__.TestC6OrphanKeys.test_unpadded_id_not_guessed) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


### <a id="c7"></a>C7 · Bad rows dropped silently

**Trap.** Filtering bad rows out with `df_orders[mask]` throws them away. The totals look fine, nobody knows what was lost, and the source team never hears about it.

**Handling.** Run every rule, join the reasons for each row (one row can have several), and split the frame:
- **quarantine**: the *raw text* exactly as it arrived, plus `reject_reason` and `source_row`, all `str`
- **valid**: the typed columns

Invariant: `rows in == valid + quarantined + exact duplicates`.

In [33]:
ORDER_RULES = [rules_duplicates, rules_parsing, rules_business, rules_reference]


def split_valid_quarantine(df_orders, df_orders_typed, df_customers) -> tuple[pd.DataFrame, pd.DataFrame]:
    checks = {}
    for rule in ORDER_RULES:
        checks |= rule(df_orders, df_orders_typed, df_customers)

    reason = pd.Series("", index=df_orders.index, dtype="str")
    for name, mask in checks.items():
        reason = reason.where(~mask, reason + name + ";")
    reason = reason.str.rstrip(";")
    bad = reason.ne("")

    df_orders_quarantine = df_orders.loc[bad].assign(reject_reason=reason[bad]).reset_index(drop=True)

    df_orders_valid = df_orders.loc[~bad, ["source_row", "order_id", "customer_id", "status", "notes"]].copy()
    df_orders_valid["order_date"] = df_orders_typed.loc[~bad, "order_date"]
    df_orders_valid["amount"] = df_orders_typed.loc[~bad, "amount"].round(2)
    df_orders_valid["quantity"] = df_orders_typed.loc[~bad, "quantity"]
    df_orders_valid["notes"] = _blank_to_na(df_orders_valid["notes"])
    return df_orders_valid, df_orders_quarantine


df_orders_valid_c7, df_orders_quarantine = split_valid_quarantine(df_orders_dedup, df_orders_typed, df_customers)
print(f"in={len(df_orders_raw)}  valid={len(df_orders_valid_c7)}  "
      f"quarantined={len(df_orders_quarantine)}  exact_dups={n_exact}")
df_orders_quarantine[["source_row", "order_id", "order_date", "amount", "quantity", "status", "reject_reason"]]

in=20  valid=8  quarantined=11  exact_dups=1


,source_row,order_id,order_date,amount,quantity,status,reject_reason
0,6,1005,2024-02-30,20.00,1,shipped,unparseable_order_date
1,7,1006,2024-03-03,N/A,1,shipped,unparseable_amount
2,8,1007,2024-03-03,15.00,1,shipped,unknown_customer
3,10,1009,,8.00,1,shipped,missing_order_date
4,11,1010,2024-03-04,9.99,,shipped,missing_quantity
5,12,1011,2024-03-05,10.00,two,shipped,unparseable_quantity
6,15,1013,2024-03-05,100.00,1,shipped,conflicting_duplicate_order_id
7,16,1013,2024-03-05,110.00,1,shipped,conflicting_duplicate_order_id
8,17,1014,2024-03-06,5.00,1,cancelled,unknown_customer
9,18,1015,2024-03-06,3.00,1,lost_in_space,invalid_status


In [34]:
class TestC7Quarantine(unittest.TestCase):
    df_orders_raw = read_raw(RAW / "orders.csv")
    df_customers = clean_customers(read_raw(RAW / "customers.csv"))
    df_orders, n_exact = drop_exact_duplicates(normalise_orders(df_orders_raw))
    df_orders_valid, df_orders_quarantine = split_valid_quarantine(df_orders, parse_orders(df_orders), df_customers)

    def test_every_input_row_accounted_for(self):
        self.assertEqual(len(self.df_orders_raw),
                         len(self.df_orders_valid) + len(self.df_orders_quarantine) + self.n_exact)

    def test_no_order_in_both_outputs(self):
        self.assertFalse(set(self.df_orders_valid["order_id"]) & set(self.df_orders_quarantine["order_id"]))

    def test_every_quarantined_row_has_a_reason(self):
        self.assertTrue(self.df_orders_quarantine["reject_reason"].str.len().gt(0).all())

    def test_quarantine_keeps_raw_text(self):
        self.assertEqual(self.df_orders_quarantine.set_index("order_id").loc["1006", "amount"], "N/A")
        for col in ORDER_COLUMNS + ["reject_reason"]:
            self.assertTrue(ptypes.is_string_dtype(self.df_orders_quarantine[col]), col)

    def test_valid_rows_complete(self):
        required = ["order_id", "customer_id", "order_date", "amount", "quantity", "status"]
        self.assertEqual(self.df_orders_valid[required].isna().sum().sum(), 0)


check(TestC7Quarantine)

test_every_input_row_accounted_for (__main__.TestC7Quarantine.test_every_input_row_accounted_for) ... 

ok


test_every_quarantined_row_has_a_reason (__main__.TestC7Quarantine.test_every_quarantined_row_has_a_reason) ... 

ok


test_no_order_in_both_outputs (__main__.TestC7Quarantine.test_no_order_in_both_outputs) ... 

ok


test_quarantine_keeps_raw_text (__main__.TestC7Quarantine.test_quarantine_keeps_raw_text) ... 

ok


test_valid_rows_complete (__main__.TestC7Quarantine.test_valid_rows_complete) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.004s

OK


### <a id="c8"></a>C8 · Join fan-out

**Trap.** If the customer table has a duplicate key, a `merge` quietly returns **more** rows than went in. Each extra row is a copied order, and revenue goes up with no error.

**Handling.** `validate="many_to_one"` makes pandas raise if the right side is not unique on the key. `how="left"` keeps orders whose customer has no country.

In [35]:
# See the trap: join to customers *before* B3's dedup and the order count grows
df_customers_with_dupes = tidy_customer_text(df_customers_raw)
df_orders_fanned = df_orders_valid_c7.merge(df_customers_with_dupes[["customer_id", "country"]],
                                            on="customer_id", how="left")
print(f"orders before join: {len(df_orders_valid_c7)}, after: {len(df_orders_fanned)}")

orders before join: 8, after: 11


In [36]:
def add_country(df_orders_valid: pd.DataFrame, df_customers: pd.DataFrame) -> pd.DataFrame:
    return df_orders_valid.merge(
        df_customers[["customer_id", "country"]], on="customer_id", how="left", validate="many_to_one"
    )


df_orders_joined = add_country(df_orders_valid_c7, df_customers)
df_orders_joined

,source_row,order_id,customer_id,status,notes,order_date,amount,quantity,country
0,2,1001,00017,shipped,NaN,2024-03-01,12.50,1.0,GB
1,3,1002,00042,shipped,"gift wrap, urgent",2024-03-01,1200.00,2.0,GB
2,4,1003,00073,delivered,NaN,2024-03-02,30.00,3.0,ES
3,5,1004,00088,delivered,NaN,2024-03-02,45.00,1.0,NA
4,9,1008,00042,refunded,NaN,2024-03-04,-5.00,1.0,GB
5,13,1012,00042,shipped,NaN,2024-03-05,7.25,1.0,GB
6,20,1017,00105,shipped,NaN,2024-03-06,4.00,1.0,NaN
7,21,1018,00073,cancelled,NaN,2024-03-06,50.00,1.0,ES


In [37]:
class TestC8JoinFanOut(unittest.TestCase):
    df_customers_raw = read_raw(RAW / "customers.csv")
    df_customers = clean_customers(df_customers_raw)
    df_orders, _ = drop_exact_duplicates(normalise_orders(read_raw(RAW / "orders.csv")))
    df_orders_valid, _ = split_valid_quarantine(df_orders, parse_orders(df_orders), df_customers)
    df_orders_joined = add_country(df_orders_valid, df_customers)

    def test_duplicate_key_raises(self):
        df_customers_with_dupes = tidy_customer_text(self.df_customers_raw)
        with self.assertRaises(pd.errors.MergeError):
            add_country(self.df_orders_valid, df_customers_with_dupes)

    def test_no_fan_out(self):
        self.assertEqual(len(self.df_orders_joined), len(self.df_orders_valid))
        self.assertTrue(self.df_orders_joined["order_id"].is_unique)

    def test_left_join_keeps_unknown_country(self):
        self.assertTrue(pd.isna(self.df_orders_joined.set_index("order_id").loc["1017", "country"]))


check(TestC8JoinFanOut)

test_duplicate_key_raises (__main__.TestC8JoinFanOut.test_duplicate_key_raises) ... 

ok


test_left_join_keeps_unknown_country (__main__.TestC8JoinFanOut.test_left_join_keeps_unknown_country) ... 

ok


test_no_fan_out (__main__.TestC8JoinFanOut.test_no_fan_out) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


### <a id="c9"></a>C9 · Order output types are not enforced

**Trap.** After parsing, splitting and joining, the valid orders' types are whatever the operations left behind:
- `quantity` is `float64`, so `1` is written as `1.0`
- `status` is free text
- the dates use the pandas default resolution
- if today's file has no notes at all, `notes` could come out as a float column

The next system sees a different schema from one day to the next.

**Handling.** `clean_orders` ends with `enforce_schema(..., ORDER_SCHEMA)`:
- `quantity` becomes `Int64`
- `status` becomes a category of the allowed values
- dates are pinned to `[ns]`
- columns are selected and ordered

C5 already quarantined fractional quantities and unknown statuses, so the cast cannot fail on valid rows. If it ever does, that is a bug, and it should stop the run.

In [38]:
def clean_orders(df_orders_raw: pd.DataFrame, df_customers: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, int]:
    """Stage C: return (df_orders_valid, df_orders_quarantine, exact_duplicates_dropped)."""
    df_orders = normalise_orders(df_orders_raw)                                   # C1
    df_orders, n_exact = drop_exact_duplicates(df_orders)                         # C2
    df_orders_typed = parse_orders(df_orders)                                     # C3
    df_orders_valid, df_orders_quarantine = split_valid_quarantine(               # C3-C7
        df_orders, df_orders_typed, df_customers)
    df_orders_valid = add_country(df_orders_valid, df_customers)                  # C8
    df_orders_valid = enforce_schema(df_orders_valid, ORDER_SCHEMA)               # C9
    return df_orders_valid.sort_values("order_id", ignore_index=True), df_orders_quarantine, n_exact


print("before C9:", schema_diff(df_orders_joined, ORDER_SCHEMA))
df_orders_valid, df_orders_quarantine, n_exact = clean_orders(df_orders_raw, df_customers)
print("after C9: ", schema_diff(df_orders_valid, ORDER_SCHEMA))
df_orders_valid

before C9: {'order_date': ('datetime64[ns]', 'datetime64[us]'), 'quantity': ('Int64', 'float64'), 'status': ('category', 'str')}
after C9:  {}


,order_id,customer_id,country,order_date,amount,quantity,status,notes,source_row
0,1001,00017,GB,2024-03-01,12.50,1,shipped,NaN,2
1,1002,00042,GB,2024-03-01,1200.00,2,shipped,"gift wrap, urgent",3
2,1003,00073,ES,2024-03-02,30.00,3,delivered,NaN,4
3,1004,00088,NA,2024-03-02,45.00,1,delivered,NaN,5
4,1008,00042,GB,2024-03-04,-5.00,1,refunded,NaN,9
5,1012,00042,GB,2024-03-05,7.25,1,shipped,NaN,13
6,1017,00105,NaN,2024-03-06,4.00,1,shipped,NaN,20
7,1018,00073,ES,2024-03-06,50.00,1,cancelled,NaN,21


In [39]:
class TestC9OrderSchema(unittest.TestCase):
    df_orders_valid, _, _ = clean_orders(read_raw(RAW / "orders.csv"), clean_customers(read_raw(RAW / "customers.csv")))

    def test_matches_schema_exactly(self):
        self.assertEqual(schema_diff(self.df_orders_valid, ORDER_SCHEMA), {})

    def test_quantity_is_whole_numbers(self):
        self.assertEqual(self.df_orders_valid["quantity"].tolist(), [1, 2, 3, 1, 1, 1, 1, 1])

    def test_status_categories_are_the_allowed_set(self):
        self.assertEqual(set(self.df_orders_valid["status"].cat.categories), VALID_STATUSES)

    def test_amount_is_whole_pennies(self):
        pennies = self.df_orders_valid["amount"] * 100
        self.assertTrue((pennies - pennies.round()).abs().lt(1e-9).all())

    def test_ids_kept_as_text(self):
        self.assertIn("00042", set(self.df_orders_valid["customer_id"]))


check(TestC9OrderSchema)

test_amount_is_whole_pennies (__main__.TestC9OrderSchema.test_amount_is_whole_pennies) ... 

ok


test_ids_kept_as_text (__main__.TestC9OrderSchema.test_ids_kept_as_text) ... 

ok


test_matches_schema_exactly (__main__.TestC9OrderSchema.test_matches_schema_exactly) ... 

ok


test_quantity_is_whole_numbers (__main__.TestC9OrderSchema.test_quantity_is_whole_numbers) ... 

ok


test_status_categories_are_the_allowed_set (__main__.TestC9OrderSchema.test_status_categories_are_the_allowed_set) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.003s

OK


---
## <a id="stage-d"></a>Stage D · Aggregate daily revenue

Goal: revenue and order count per day and country, matching `DAILY_REVENUE_SCHEMA` and adding up to exactly the billable valid orders.

### <a id="d1"></a>D1 · Cancelled orders counted as revenue

**Trap.** Cancelled orders are valid data (they stay in `orders_clean`) but they are not revenue. Refunds *are* revenue: negative amounts that reduce the total.

**Handling.** Aggregate only billable orders: everything except `cancelled`.

In [40]:
def billable_orders(df_orders_valid: pd.DataFrame) -> pd.DataFrame:
    return df_orders_valid.loc[df_orders_valid["status"].ne("cancelled")]

In [41]:
class TestD1Billable(unittest.TestCase):
    df_orders_valid, _, _ = clean_orders(read_raw(RAW / "orders.csv"), clean_customers(read_raw(RAW / "customers.csv")))
    df_orders_billable = billable_orders(df_orders_valid)

    def test_cancelled_excluded(self):
        self.assertNotIn("cancelled", set(self.df_orders_billable["status"]))
        self.assertNotIn("1018", set(self.df_orders_billable["order_id"]))

    def test_refunds_kept(self):
        self.assertIn("1008", set(self.df_orders_billable["order_id"]))


check(TestD1Billable)

test_cancelled_excluded (__main__.TestD1Billable.test_cancelled_excluded) ... 

ok


test_refunds_kept (__main__.TestD1Billable.test_refunds_kept) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


### <a id="d2"></a>D2 · `groupby` drops NaN keys

**Trap.** Customer `00105` has no country. `groupby` drops NaN keys by default, so that revenue disappears from the report with no warning.

**Handling.** `groupby(..., dropna=False)` keeps an unknown-country group. Revenue is rounded at the end so float noise does not leak out, and the result is cast to `DAILY_REVENUE_SCHEMA`.

In [42]:
# See the trap: default groupby loses revenue
df_orders_billable = billable_orders(df_orders_valid)
print("billable total:", df_orders_billable["amount"].sum(),
      "| default groupby total:", df_orders_billable.groupby(["order_date", "country"])["amount"].sum().sum())

billable total: 1293.75 | default groupby total: 1289.75


In [43]:
def daily_revenue(df_orders_valid: pd.DataFrame) -> pd.DataFrame:
    """Stage D: revenue and order count per day and country, declared types."""
    df_daily_revenue = (
        billable_orders(df_orders_valid)
        .groupby(["order_date", "country"], dropna=False)
        .agg(revenue=("amount", "sum"), orders=("order_id", "nunique"))
        .reset_index()
        .sort_values(["order_date", "country"], na_position="last", ignore_index=True)
    )
    df_daily_revenue["revenue"] = df_daily_revenue["revenue"].round(2)
    return enforce_schema(df_daily_revenue, DAILY_REVENUE_SCHEMA)


df_daily_revenue = daily_revenue(df_orders_valid)
df_daily_revenue

,order_date,country,revenue,orders
0,2024-03-01,GB,1212.50,2
1,2024-03-02,ES,30.00,1
2,2024-03-02,NA,45.00,1
3,2024-03-04,GB,-5.00,1
4,2024-03-05,GB,7.25,1
5,2024-03-06,NaN,4.00,1


In [44]:
class TestD2Aggregate(unittest.TestCase):
    df_orders_valid, _, _ = clean_orders(read_raw(RAW / "orders.csv"), clean_customers(read_raw(RAW / "customers.csv")))
    df_daily_revenue = daily_revenue(df_orders_valid)

    def test_total_reconciles_with_billable_orders(self):
        df_orders_billable = billable_orders(self.df_orders_valid)
        self.assertAlmostEqual(self.df_daily_revenue["revenue"].sum(), df_orders_billable["amount"].sum(), places=2)
        self.assertEqual(self.df_daily_revenue["orders"].sum(), len(df_orders_billable))

    def test_unknown_country_group_kept(self):
        self.assertTrue(self.df_daily_revenue["country"].isna().any())

    def test_matches_schema_exactly(self):
        self.assertEqual(schema_diff(self.df_daily_revenue, DAILY_REVENUE_SCHEMA), {})

    def test_tidy_output(self):
        self.assertIsInstance(self.df_daily_revenue.index, pd.RangeIndex)
        self.assertTrue(self.df_daily_revenue["order_date"].is_monotonic_increasing)
        self.assertFalse(self.df_daily_revenue.duplicated(["order_date", "country"]).any())


check(TestD2Aggregate)

test_matches_schema_exactly (__main__.TestD2Aggregate.test_matches_schema_exactly)

 ... 

ok


test_tidy_output (__main__.TestD2Aggregate.test_tidy_output) ... 

ok


test_total_reconciles_with_billable_orders (__main__.TestD2Aggregate.test_total_reconciles_with_billable_orders) ... 

ok


test_unknown_country_group_kept (__main__.TestD2Aggregate.test_unknown_country_group_kept) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


---
## <a id="stage-e"></a>Stage E · Publish

Goal: write outputs that can be trusted and rerun: every row accounted for, rejections visible, the same files every time, and types that survive the trip to disk.

### <a id="e1"></a>E1 · Rows vanish between input and output

**Trap.** A bug in any stage (a bad filter, a wrong join) can lose rows, and the outputs still look plausible.

**Handling.** Before publishing, check that `rows in == valid + quarantined + exact duplicates`. If not, raise and write nothing.

*Tests vs checks:* the tests in this notebook prove the code is right on the sample files. This check runs on **every real run** and protects the data.

In [45]:
def reconcile(n_in: int, df_orders_valid: pd.DataFrame, df_orders_quarantine: pd.DataFrame, n_exact: int) -> None:
    accounted = len(df_orders_valid) + len(df_orders_quarantine) + n_exact
    if accounted != n_in:
        raise RuntimeError(f"row reconciliation failed: {n_in} in, {accounted} out")


reconcile(len(df_orders_raw), df_orders_valid, df_orders_quarantine, n_exact)
print("reconciled:", len(df_orders_raw), "=", len(df_orders_valid), "+", len(df_orders_quarantine), "+", n_exact)

reconciled: 20 = 8 + 11 + 1


In [46]:
class TestE1Reconcile(unittest.TestCase):
    df_orders_raw = read_raw(RAW / "orders.csv")
    df_orders_valid, df_orders_quarantine, n_exact = clean_orders(df_orders_raw, clean_customers(read_raw(RAW / "customers.csv")))

    def test_passes_on_sample(self):
        reconcile(len(self.df_orders_raw), self.df_orders_valid, self.df_orders_quarantine, self.n_exact)

    def test_lost_row_raises(self):
        with self.assertRaises(RuntimeError):
            reconcile(len(self.df_orders_raw), self.df_orders_valid.iloc[1:], self.df_orders_quarantine, self.n_exact)


check(TestE1Reconcile)

test_lost_row_raises (__main__.TestE1Reconcile.test_lost_row_raises) ... 

ok


test_passes_on_sample (__main__.TestE1Reconcile.test_passes_on_sample) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


### <a id="e2"></a>E2 · Nobody can see what was rejected

**Trap.** The quarantine file exists, but nobody opens a CSV to find out whether today's run was healthy.

**Handling.** Write a small `run_report.json` with rows in and out and a count per reject reason. It can be diffed between runs, alerted on, or read in a second. Counts are cast to plain `int` and totals to `float`, because numpy types are not JSON-serialisable.

In [47]:
def build_report(df_customers_raw, df_customers, df_orders_raw, df_orders_valid,
                 df_orders_quarantine, n_exact, df_daily_revenue) -> dict:
    reasons = df_orders_quarantine["reject_reason"].str.split(";").explode().value_counts().sort_index()
    return {
        "customers": {"rows_in": len(df_customers_raw), "rows_out": len(df_customers)},
        "orders": {
            "rows_in": len(df_orders_raw),
            "valid": len(df_orders_valid),
            "quarantined": len(df_orders_quarantine),
            "exact_duplicates_dropped": n_exact,
            "reject_reasons": {k: int(v) for k, v in reasons.items()},
        },
        "revenue_total": round(float(df_daily_revenue["revenue"].sum()), 2),
    }


build_report(df_customers_raw, df_customers, df_orders_raw, df_orders_valid,
             df_orders_quarantine, n_exact, df_daily_revenue)

{'customers': {'rows_in': 8, 'rows_out': 6},
 'orders': {'rows_in': 20,
  'valid': 8,
  'quarantined': 11,
  'exact_duplicates_dropped': 1,
  'reject_reasons': {'conflicting_duplicate_order_id': 2,
   'invalid_status': 1,
   'missing_order_date': 1,
   'missing_quantity': 1,
   'unknown_customer': 2,
   'unparseable_amount': 1,
   'unparseable_order_date': 2,
   'unparseable_quantity': 1}},
 'revenue_total': 1293.75}

In [48]:
class TestE2Report(unittest.TestCase):
    df_customers_raw = read_raw(RAW / "customers.csv")
    df_orders_raw = read_raw(RAW / "orders.csv")
    df_customers = clean_customers(df_customers_raw)
    df_orders_valid, df_orders_quarantine, n_exact = clean_orders(df_orders_raw, df_customers)
    report = build_report(df_customers_raw, df_customers, df_orders_raw, df_orders_valid,
                          df_orders_quarantine, n_exact, daily_revenue(df_orders_valid))

    def test_counts_add_up(self):
        o = self.report["orders"]
        self.assertEqual(o["rows_in"], o["valid"] + o["quarantined"] + o["exact_duplicates_dropped"])

    def test_every_rejection_counted(self):
        self.assertGreaterEqual(sum(self.report["orders"]["reject_reasons"].values()), len(self.df_orders_quarantine))

    def test_json_safe(self):
        self.assertEqual(json.loads(json.dumps(self.report)), self.report)


check(TestE2Report)

test_counts_add_up (__main__.TestE2Report.test_counts_add_up) ... 

ok


test_every_rejection_counted (__main__.TestE2Report.test_every_rejection_counted) ... 

ok


test_json_safe (__main__.TestE2Report.test_json_safe) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.001s

OK


### <a id="e3"></a>E3 · Reruns append or leave stale files

**Trap.** Batch jobs get rerun after failures and backfills. If outputs are appended, or file names depend on the time, a rerun doubles the data or leaves stale files behind.

**Handling.** Always overwrite the same fixed set of files, and keep the output stable (sorted rows, sorted JSON keys). Running twice gives byte-identical files. `run()` wires stages A to E together.

No global `date_format` is passed to `to_csv`. A `"%Y-%m-%d"` format would silently cut the time off `updated_at`. pandas already writes a date-only column as `YYYY-MM-DD`.

In [49]:
def write_outputs(out_dir: Path, tables: dict[str, pd.DataFrame], report: dict) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, df_table in tables.items():
        df_table.to_csv(out_dir / f"{name}.csv", index=False)
    (out_dir / "run_report.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")


def run(raw_dir: Path = RAW, out_dir: Path = OUT_DIR) -> dict:
    raw_dir, out_dir = Path(raw_dir), Path(out_dir)
    df_customers_raw = read_raw(raw_dir / "customers.csv")                              # A
    df_orders_raw = read_raw(raw_dir / "orders.csv")                                    # A
    df_customers = clean_customers(df_customers_raw)                                    # B
    df_orders_valid, df_orders_quarantine, n_exact = clean_orders(df_orders_raw, df_customers)  # C
    reconcile(len(df_orders_raw), df_orders_valid, df_orders_quarantine, n_exact)       # E1
    df_daily_revenue = daily_revenue(df_orders_valid)                                   # D
    report = build_report(df_customers_raw, df_customers, df_orders_raw, df_orders_valid,
                          df_orders_quarantine, n_exact, df_daily_revenue)              # E2
    write_outputs(out_dir, {                                                            # E3
        "customers_clean": df_customers,
        "orders_clean": df_orders_valid,
        "orders_quarantine": df_orders_quarantine,
        "daily_revenue": df_daily_revenue,
    }, report)
    return report


report = run()
report

{'customers': {'rows_in': 8, 'rows_out': 6},
 'orders': {'rows_in': 20,
  'valid': 8,
  'quarantined': 11,
  'exact_duplicates_dropped': 1,
  'reject_reasons': {'conflicting_duplicate_order_id': 2,
   'invalid_status': 1,
   'missing_order_date': 1,
   'missing_quantity': 1,
   'unknown_customer': 2,
   'unparseable_amount': 1,
   'unparseable_order_date': 2,
   'unparseable_quantity': 1}},
 'revenue_total': 1293.75}

In [50]:
class TestE3Idempotent(unittest.TestCase):
    def test_rerun_gives_identical_files(self):
        with tempfile.TemporaryDirectory() as a, tempfile.TemporaryDirectory() as b:
            run(RAW, Path(a))
            run(RAW, Path(a))  # second run over the same output dir
            run(RAW, Path(b))
            files = sorted(p.name for p in Path(a).iterdir())
            _, mismatch, errors = filecmp.cmpfiles(a, b, files, shallow=False)
            self.assertEqual((mismatch, errors), ([], []))
            self.assertEqual(sorted(p.name for p in Path(b).iterdir()), files)  # no stray files

    def test_report_on_disk_matches(self):
        with tempfile.TemporaryDirectory() as d:
            report = run(RAW, Path(d))
            self.assertEqual(json.loads((Path(d) / "run_report.json").read_text()), report)


check(TestE3Idempotent)

test_report_on_disk_matches (__main__.TestE3Idempotent.test_report_on_disk_matches) ... 

ok


test_rerun_gives_identical_files (__main__.TestE3Idempotent.test_rerun_gives_identical_files) ... 

ok

----------------------------------------------------------------------
Ran 2 tests in 0.060s

OK


### <a id="e4"></a>E4 · Types are lost when the output is written to CSV

**Trap.** CSV has no types. Whoever reads `orders_clean.csv` with a plain `read_csv` is back at A1:
- `customer_id` `00042` becomes `42`
- Namibia's `NA` becomes NaN
- dates come back as text
- `status` is no longer a category

The pipeline's careful types exist only in memory.

**Handling.** Publish the schemas with the data, and read outputs back through `read_output`. It reads everything as text, treats only `""` as missing, parses dates and numbers strictly, and then calls `enforce_schema`. The test is a **round trip**: write, read back, and compare with the in-memory frame, value for value and dtype for dtype. In production, Parquet stores the types in the file itself.

In [51]:
# See the trap: a plain read of what we just published
df_orders_naive = pd.read_csv(OUT_DIR / "orders_clean.csv")
print(df_orders_naive.dtypes.astype(str).to_dict())
df_orders_naive[["order_id", "customer_id", "country", "order_date", "status"]].head(4)

{'order_id': 'int64', 'customer_id': 'int64', 'country': 'str', 'order_date': 'str', 'amount': 'float64', 'quantity': 'int64', 'status': 'str', 'notes': 'str', 'source_row': 'int64'}


,order_id,customer_id,country,order_date,status
0,1001,17,GB,2024-03-01,shipped
1,1002,42,GB,2024-03-01,shipped
2,1003,73,ES,2024-03-02,delivered
3,1004,88,NaN,2024-03-02,delivered


In [52]:
def read_output(path: Path, schema: dict) -> pd.DataFrame:
    """Works on any published table: read a CSV back with its declared schema; never guess."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False)    # same rules as A1
    df = df.mask(df.eq(""))                                     # only a blank cell is missing ("NA" stays Namibia)
    for col, dtype in schema.items():
        if str(dtype).startswith("datetime64"):
            df[col] = pd.to_datetime(df[col], format="ISO8601")  # strict ISO; raises on anything else
        elif str(dtype) in {"float64", "Int64", "int64"}:
            df[col] = pd.to_numeric(df[col])                    # raises on non-numbers
    return enforce_schema(df, schema)


OUTPUT_SCHEMAS = {
    "customers_clean": CUSTOMER_SCHEMA,
    "orders_clean": ORDER_SCHEMA,
    "daily_revenue": DAILY_REVENUE_SCHEMA,
}
df_orders_reread = read_output(OUT_DIR / "orders_clean.csv", ORDER_SCHEMA)
print(schema_diff(df_orders_reread, ORDER_SCHEMA))
df_orders_reread[["order_id", "customer_id", "country", "order_date", "status"]].head(4)

{}


,order_id,customer_id,country,order_date,status
0,1001,00017,GB,2024-03-01,shipped
1,1002,00042,GB,2024-03-01,shipped
2,1003,00073,ES,2024-03-02,delivered
3,1004,00088,NA,2024-03-02,delivered


In [53]:
class TestE4RoundTrip(unittest.TestCase):
    def test_outputs_read_back_identical(self):
        df_customers = clean_customers(read_raw(RAW / "customers.csv"))
        df_orders_valid, _, _ = clean_orders(read_raw(RAW / "orders.csv"), df_customers)
        in_memory = {
            "customers_clean": df_customers,
            "orders_clean": df_orders_valid,
            "daily_revenue": daily_revenue(df_orders_valid),
        }
        with tempfile.TemporaryDirectory() as d:
            run(RAW, Path(d))
            for name, schema in OUTPUT_SCHEMAS.items():
                with self.subTest(table=name):
                    pd.testing.assert_frame_equal(read_output(Path(d) / f"{name}.csv", schema), in_memory[name])

    def test_timestamp_time_survives(self):
        with tempfile.TemporaryDirectory() as d:
            run(RAW, Path(d))
            df_customers = read_output(Path(d) / "customers_clean.csv", CUSTOMER_SCHEMA)
        updated = df_customers.set_index("customer_id").loc["00042", "updated_at"]
        self.assertEqual(updated, pd.Timestamp("2024-03-10 12:30:00"))


check(TestE4RoundTrip)

test_outputs_read_back_identical (__main__.TestE4RoundTrip.test_outputs_read_back_identical) ... 

ok


test_timestamp_time_survives (__main__.TestE4RoundTrip.test_timestamp_time_survives) ... 

ok


----------------------------------------------------------------------
Ran 2 tests in 0.050s

OK


---
## <a id="all-tests"></a>All tests

Every step's test class again, in one run. Use it after changing any function above.

In [54]:
check(*ALL_TESTS.values())
print(f"\n{len(ALL_TESTS)} test classes:", ", ".join(ALL_TESTS))

test_every_column_is_text (__main__.TestA1NothingGuessed.test_every_column_is_text) ... 

ok


test_leading_zeros_preserved (__main__.TestA1NothingGuessed.test_leading_zeros_preserved) ... 

ok


test_literal_NA_is_not_missing (__main__.TestA1NothingGuessed.test_literal_NA_is_not_missing) ... 

ok


... (124 outputs left out)

test_timestamp_time_survives (__main__.TestE4RoundTrip.test_timestamp_time_survives) ... 

ok


----------------------------------------------------------------------
Ran 66 tests in 0.179s

OK



22 test classes: TestA1NothingGuessed, TestA2Lineage, TestA3Schema, TestB1TidyText, TestB2StrictDates, TestB3LatestVersion, TestB4CustomerSchema, TestC1Normalise, TestC2ExactDuplicates, TestC3Parsing, TestC4ConflictingDuplicates, TestC5BusinessRules, TestC6OrphanKeys, TestC7Quarantine, TestC8JoinFanOut, TestC9OrderSchema, TestD1Billable, TestD2Aggregate, TestE1Reconcile, TestE2Report, TestE3Idempotent, TestE4RoundTrip


## <a id="notes"></a>Notes (the kind of thing to write in the markdown cell they ask for)

**Assumptions**
- `amount` is the order total, not a unit price (would ask). Currency symbols and thousands separators are stripped and everything is assumed to be GBP.
- Only ISO dates (`YYYY-MM-DD`) are accepted. `06/03/2024` is ambiguous (6 March or 3 June), so it goes to quarantine and is not guessed.
- Customer `17` is probably `00017` with its zeros lost in Excel. I quarantined it instead of padding, because a wrong guess would put revenue on the wrong customer.

**Decisions**
- Exact duplicate rows are dropped and counted. Rows that share an `order_id` but differ in values are *both* quarantined, because I can't tell which one is true.
- A negative amount is valid only for `refunded`. Quantities must be whole and positive. Cancelled orders stay in the clean data but are excluded from revenue.
- Customers: the latest `updated_at` wins.
- Types:
  - read everything as text
  - parse strictly
  - declare one schema per output (`CUSTOMER_SCHEMA`, `ORDER_SCHEMA`, `DAILY_REVENUE_SCHEMA`) and enforce it at the end of each stage
  - ids are `str`, `quantity` is nullable `Int64`, `status` is a category, dates are `datetime64[ns]`, money is `float64` rounded to pennies

**What downstream sees**: `orders_quarantine.csv` keeps the raw text plus `reject_reason` and `source_row`. `run_report.json` holds the counts per reason, and a reconciliation check (in = valid + quarantined + dups) stops the run if it fails. CSV outputs must be read back with their schema (`read_output`).

**Tests vs checks**: The tests prove the code is right on this sample. On every real run, the quarantine, the schema enforcement and the reconciliation check inside `run()` are what guard the data.

**Next with more time** (left out on purpose to keep the notebook small): money as integer pence or `Decimal`; Parquet so types travel with the file; atomic writes; failing the run above a quarantine-rate threshold; partitioning by date; incremental loads by `order_date`.